
# 04 - Cleaning precipitation dataset
**Goal:** Clean the dataset to develop

**Outputs:**
- Clean invalid GOES images points
- Define partition strategy
- *The final dataset is created with scripts* `scripts/dataset/03_XX` -> `data/processed/XXXXX.csv`

**Next:** Based on findings, define cleaning rules in `04_cleaning_rules_decisions.ipynb`.


### Import Libraries

In [14]:
# Standard library imports
from pathlib import Path

# Third-party imports
import pandas as pd

# Local application imports
from dlgoes.utils.config import load_config_ns
from dlgoes.utils.seed import set_seed
from dlgoes.utils.config import find_repo_root
from dlgoes.data.precipitation.clean import comprobar_frames
from dlgoes.data.precipitation.clean import changeOrigenStation
from dlgoes.data.precipitation.clean import clean_dataset
from dlgoes.data.precipitation.split import split_dataset


repo_root = find_repo_root(Path.cwd())
cfg = load_config_ns(repo_root / 'configs' / 'base.yaml')
clean_cfg = load_config_ns(repo_root / 'configs' / 'cleaning' /'base.yaml')
set_seed(cfg.seed)

### Funciones

In [15]:
import plotly.express as px

def graficar(dfbase, color_col='target'):
    print('Tamaño:', len(dfbase))
    print('Estaciones:', dfbase['CODE'].nunique())

    fig = px.scatter_map(
        dfbase,
        lon='LON',
        lat='LAT',
        color=color_col,
        hover_name='CODE',
        hover_data={color_col: True},
        zoom=5,
        height=700,
        color_discrete_sequence=px.colors.qualitative.Set2
    )

    fig.update_layout(
        map_style="carto-positron",  # 👈 cambia de mapbox_style
        margin=dict(l=0, r=0, t=40, b=0),
        title=f"Mapa de estaciones ({color_col})"
    )

    fig.show()

### Read Data

In [16]:
# Read the precipitation dataset
df = pd.read_csv(cfg.paths.project_root / cfg.paths.data.interim/clean_cfg.input_data, index_col=0)
df.head()

,CODE,NOMBRE,FECHA,HORA,PRECIPITACION,FLAG,FLAG_V2,ESTACION,LON,LAT,ALT,THS_1,THS_2,MIN_1,MIN_2
0,X4722A338,ACJANACO,03/04/2020,00:00:00,0.0,C0000001,C01,ACJANACO,-71.61941,-13.19672,3466.0,1.5,9.7,0.0,0.0
1,X4722A338,ACJANACO,03/04/2020,01:00:00,0.0,C0000001,C01,ACJANACO,-71.61941,-13.19672,3466.0,1.5,9.7,0.0,0.0
2,X4722A338,ACJANACO,03/04/2020,02:00:00,0.0,C0000001,C01,ACJANACO,-71.61941,-13.19672,3466.0,1.5,9.7,0.0,0.0
3,X4722A338,ACJANACO,03/04/2020,03:00:00,0.0,C0000001,C01,ACJANACO,-71.61941,-13.19672,3466.0,1.5,9.7,0.0,0.0
4,X4722A338,ACJANACO,03/04/2020,04:00:00,0.0,C0000001,C01,ACJANACO,-71.61941,-13.19672,3466.0,1.5,9.7,0.0,0.0


In [17]:
df["_FECHA_HORA"] = pd.to_datetime(
    df["FECHA"].astype(str) + " " + df["HORA"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

In [18]:
# Marcar flags (MISSING_GOES): 1 si falta, 0 si existe
fechas_faltantes = comprobar_frames(cfg, df)
df['_MISSING_GOES'] = 0
df = changeOrigenStation(cfg, df)

df.head(2)

251


,CODE,NOMBRE,FECHA,HORA,PRECIPITACION,FLAG,FLAG_V2,ESTACION,LON,LAT,ALT,THS_1,THS_2,MIN_1,MIN_2,_FECHA_HORA,_MISSING_GOES,XLO,XLA
0,X4722A338,ACJANACO,03/04/2020,00:00:00,0.0,C0000001,C01,ACJANACO,-71.61941,-13.19672,3466.0,1.5,9.7,0.0,0.0,2020-04-03 00:00:00,0,662,809
1,X4722A338,ACJANACO,03/04/2020,01:00:00,0.0,C0000001,C01,ACJANACO,-71.61941,-13.19672,3466.0,1.5,9.7,0.0,0.0,2020-04-03 01:00:00,0,662,809


In [19]:
df_valid = clean_dataset(clean_cfg, df)
df_valid.head(2)

,CODE,NOMBRE,FECHA,HORA,PRECIPITACION,FLAG,FLAG_V2,ESTACION,LON,LAT,...,THS_1,THS_2,MIN_1,MIN_2,_FECHA_HORA,_MISSING_GOES,XLO,XLA,valid_data,TARGET
0,X4722A338,ACJANACO,03/04/2020,00:00:00,0.0,C0000001,C01,ACJANACO,-71.61941,-13.19672,...,1.5,9.7,0.0,0.0,2020-04-03 00:00:00,0,662,809,False,0
1,X4722A338,ACJANACO,03/04/2020,01:00:00,0.0,C0000001,C01,ACJANACO,-71.61941,-13.19672,...,1.5,9.7,0.0,0.0,2020-04-03 01:00:00,0,662,809,False,0


In [20]:
df_final = split_dataset(clean_cfg, df_valid[df_valid['valid_data']==True])

In [21]:
df_final['valid_data'].value_counts()

valid_data
True    36788
Name: count, dtype: int64

In [22]:
df_final['_MISSING_GOES'].unique()

array([0])

In [23]:
df_final.head(2)

,CODE,NOMBRE,FECHA,HORA,PRECIPITACION,FLAG,FLAG_V2,ESTACION,LON,LAT,...,MIN_2,_FECHA_HORA,_MISSING_GOES,XLO,XLA,valid_data,TARGET,_day_group,_group_id,split
0,X107131,LA FORTUNA,02/05/2021,18:00:00,11.5,C0000002,D02,LA FORTUNA,-78.40242,-7.67042,...,0.0,2021-05-02 18:00:00,0,286,502,True,1,2021-05-02,X107131-20210502,train
1,X107131,LA FORTUNA,03/05/2021,09:00:00,0.1,C0000002,D01,LA FORTUNA,-78.40242,-7.67042,...,0.0,2021-05-03 09:00:00,0,286,502,True,1,2021-05-02,X107131-20210502,train


In [24]:
df_final.groupby(['split','target']).count()

KeyError: 'target'

### Plot / Analysis final dataset

In [ ]:
graficar(df_final, color_col='split')

Tamaño: 36788
Estaciones: 164


In [ ]:
df_final.head(2)

,CODE,NOMBRE,FECHA,HORA,PRECIPITACION,FLAG,FLAG_V2,ESTACION,LON,LAT,...,MIN_2,_FECHA_HORA,_MISSING_GOES,XLO,XLA,valid_data,target,_day_group,_group_id,split
0,X107131,LA FORTUNA,02/05/2021,18:00:00,11.5,C0000002,D02,LA FORTUNA,-78.40242,-7.67042,...,0.0,2021-05-02 18:00:00,0,286,502,True,1,2021-05-02,X107131-20210502,train
1,X107131,LA FORTUNA,03/05/2021,09:00:00,0.1,C0000002,D01,LA FORTUNA,-78.40242,-7.67042,...,0.0,2021-05-03 09:00:00,0,286,502,True,1,2021-05-02,X107131-20210502,train


### Findings
- The final dataset after cleaning, shows a distribution for M02 Flag of 0.9% 
- After the split, we obtain a balanced geography distribution on the develpment datset(train,test y holdout)

### Next Actions
- Move the finalized cleaning and split rules to `src/dlgoes/data/..`  once they are defined.
- We will apply data-augmentation in the train dataset.
- Setup the pipeline for training
